<a href="https://colab.research.google.com/github/ShahadSaeedAlshahrani/NLP_Labs/blob/main/Lab12/Lab12_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers


#### A Transformer is a deep learning architecture designed to process sequences of text using self-attention.

- Transformers are the foundation of many modern NLP models, including BERT, GPT, and T5.

- Before Transformers, we used models such as RNNs and LSTMs for NLP tasks.

<img src="https://drive.google.com/thumbnail?id=1unPqGaKftHxp37-F99Bg6hJzFbQdCGC7&sz=w1000" width="700">

In [1]:
%pip -q install transformers torch matplotlib

## 1. Load a Transformer

We use **DistilBERT**, a  Transformer fine-tuned for positive/negative sentiment classification.

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
# uncased -> not case sensitive

tokenizer = AutoTokenizer.from_pretrained(model_name)
# download the tokenizer of this model

# download the model
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval() # freez the training layers, we will only use the evaluation

# Look inside the Transformer architecture.
print("Transformer layers:", model.config.n_layers)
print("Attention heads per layer:", model.config.n_heads)
print("Hidden size:", model.config.dim) # each word's vector dimention

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Transformer layers: 6
Attention heads per layer: 12
Hidden size: 768



<img src="https://drive.google.com/thumbnail?id=1v2dRK1KR_ydZOknjj8Xv8_wjOdXleDYs&sz=w1000" width="500">

## 2. Tokenizing the text

In [3]:
sentence = "The movie was not good."

# Tokenization: converts text into token IDs
inputs = tokenizer(sentence, return_tensors="pt") # and return it as a tensor

# [CLS] = start token, [SEP] = end token
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]) # convert numbers into tokens again

print("Tokens:", tokens)

Tokens: ['[CLS]', 'the', 'movie', 'was', 'not', 'good', '.', '[SEP]']


## 3. Embeddings and self-attention

Each token ID becomes a **vector (embedding)**. Inside each Transformer layer, **self-attention** lets a token use information from other tokens in the same sentence.

In [4]:
import torch

# Enable attention weights
model.set_attn_implementation("eager") # attention weights

with torch.no_grad(): # no training, we only need prediction
    embeddings = model.distilbert.embeddings(inputs["input_ids"])
    outputs = model(**inputs, output_attentions=True)
    # output_attentions=True --> returns attention weights for each layer and each head

# Attention from the first layer and first head
attention = outputs.attentions[0][0, 0]

print("Embeddings shape:", embeddings.shape)
print("Attention shape:", attention.shape)
print("Tokens:", tokens)
print("Attention weights:\n", attention)

Embeddings shape: torch.Size([1, 8, 768])
Attention shape: torch.Size([8, 8])
Tokens: ['[CLS]', 'the', 'movie', 'was', 'not', 'good', '.', '[SEP]']
Attention weights:
 tensor([[0.1145, 0.0938, 0.0681, 0.0701, 0.0596, 0.0800, 0.1593, 0.3546],
        [0.4147, 0.0793, 0.0481, 0.1018, 0.1084, 0.0805, 0.0789, 0.0883],
        [0.1634, 0.0327, 0.0975, 0.0403, 0.1819, 0.1402, 0.0866, 0.2574],
        [0.2882, 0.1014, 0.0504, 0.0669, 0.2180, 0.1009, 0.1231, 0.0511],
        [0.3612, 0.0501, 0.0696, 0.1200, 0.0642, 0.1304, 0.0953, 0.1092],
        [0.3065, 0.0469, 0.1175, 0.1915, 0.0630, 0.0555, 0.1120, 0.1071],
        [0.3505, 0.0935, 0.0572, 0.0753, 0.0972, 0.0964, 0.1165, 0.1135],
        [0.3513, 0.1019, 0.0318, 0.0541, 0.1036, 0.0607, 0.0755, 0.2210]])


## 4. Classification: From Transformer to prediction

The Transformer produces contextual representations. The **classification head** turns them into two scores (*logits*). **Softmax** converts these into probabilities that sum to 1.

In [5]:
# Get prediction scores from the Transformer
with torch.no_grad():
    logits = model(**inputs).logits # the results befor converting it into propablities

# Convert scores into probabilities
probs = torch.softmax(logits, dim=1)[0]

# Display the prediction
prediction = probs.argmax().item()

print("Prediction:", model.config.id2label[prediction]) # convert id to positive and negative
print("Confidence:", round(probs[prediction].item(), 3))


Prediction: NEGATIVE
Confidence: 1.0



### Task: Exploring Transformer Layers and Attention Heads

Load the pretrained **`bert-base-uncased`** model using Hugging Face Transformers.

1. Write code to print the number of Transformer layers and the number of attention heads per layer.
2. Calculate and print the total number of attention heads across all Transformer layers.
3. Using the sentence `"Transformers use self-attention to understand language."`, extract the attention weights from the **third Transformer layer and second attention head**.
4. Explain what the rows and columns of the attention matrix represent.

Load the pretrained **`bert-base-uncased`** model using Hugging Face Transformers.

In [15]:
from transformers import BertModel, AutoTokenizer

model_name2 = "bert-base-uncased"

tokenizer2 = AutoTokenizer.from_pretrained(model_name2)

model2 = BertModel.from_pretrained(model_name,output_attentions=True)
model2.eval()

[transformers] You are using a model of type `distilbert` to instantiate a model of type `bert`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights: 0it [00:00, ?it/s]

[transformers] BertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                                                                      | Status     | 
-------------------------------------------------------------------------+------------+-
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.k_lin.bias     | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.out_lin.bias   | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.weight   | UNEXPECTED | 
pre_classifier.weight                                                    | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.ffn.lin2.bias            | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.ffn.lin1.bias            | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.bias     | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.out_lin.weight | UNEXPECTED | 
distilbert.embeddin

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

1. Write code to print the number of Transformer layers and the number of attention heads per layer.


In [17]:
num_layers = model2.config.num_hidden_layers
num_heads = model2.config.num_attention_heads

print("Transformer layers:", num_layers)
print("Attention heads per layer:", num_heads)

Transformer layers: 12
Attention heads per layer: 12


2. Calculate and print the total number of attention heads across all Transformer layers.


In [18]:
total_heads = num_layers * num_heads

print("Total attention heads:", total_heads)

Total attention heads: 144


3. Using the sentence `"Transformers use self-attention to understand language."`, extract the attention weights from the **third Transformer layer and second attention head**.


In [19]:
sentence2 = "Transformers use self-attention to understand language."
inputs2 = tokenizer2(sentence2, return_tensors="pt")

with torch.no_grad():
  outputs2 = model2(**inputs2, output_attentions=True, return_dict=True)

tokens2 = tokenizer2.convert_ids_to_tokens(inputs2["input_ids"][0])

# Third layer -> index 2
# Second head -> index 1
attention2 = outputs2.attentions[2][0, 1]

print("\nTokens:")
print(tokens2)

print("\nAttention matrix shape:")
print(attention2.shape)

print("\nAttention weights:")
print(attention2)


Tokens:
['[CLS]', 'transformers', 'use', 'self', '-', 'attention', 'to', 'understand', 'language', '.', '[SEP]']

Attention matrix shape:
torch.Size([11, 11])

Attention weights:
tensor([[0.0836, 0.0901, 0.1070, 0.0914, 0.0985, 0.0839, 0.1014, 0.0812, 0.0849,
         0.1053, 0.0727],
        [0.1139, 0.0716, 0.0860, 0.0837, 0.0921, 0.0810, 0.0820, 0.0750, 0.0969,
         0.1096, 0.1083],
        [0.0960, 0.0947, 0.1168, 0.0965, 0.1014, 0.0684, 0.0874, 0.0890, 0.0953,
         0.0841, 0.0704],
        [0.0763, 0.1322, 0.0811, 0.0816, 0.0897, 0.0804, 0.1122, 0.0737, 0.1041,
         0.0970, 0.0716],
        [0.1067, 0.0782, 0.1334, 0.0753, 0.0969, 0.1110, 0.0880, 0.0631, 0.0663,
         0.0851, 0.0960],
        [0.1029, 0.0799, 0.1043, 0.0997, 0.0934, 0.0926, 0.1006, 0.0563, 0.0917,
         0.0958, 0.0828],
        [0.0837, 0.0828, 0.1064, 0.0610, 0.1166, 0.0950, 0.1026, 0.0684, 0.0798,
         0.1160, 0.0877],
        [0.1026, 0.0903, 0.1098, 0.0789, 0.0712, 0.0706, 0.1197, 0.0799

4. Explain what the rows and columns of the attention matrix represent.  
Rows is for 'query tokens' which are the tokens that need information  
Coulmns is for 'key tokens' which are the tokens having attention weights for each haed